# Notebook 05 - Embedding Layer

## Objetivo
Construir una red simple con `Embedding`, interpretar parametros clave y observar shapes de entrada y salida.

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

# Dataset pequeno para demostracion
textos = [
    "excelente clase de nlp",
    "contenido muy claro y util",
    "explicacion confusa y lenta",
    "laboratorio excelente y practico",
    "no me gusto la clase",
    "muy buena introduccion a embeddings"
]
etiquetas = np.array([1, 1, 0, 1, 0, 1])

## Preparar secuencias
Tokenizamos, convertimos a secuencias y aplicamos padding para longitud uniforme.

In [2]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(textos)
secuencias = tokenizer.texts_to_sequences(textos)

input_length = 6
x = pad_sequences(secuencias, maxlen=input_length, padding="post", truncating="post")

print("Shape de entrada (x):", x.shape)
print("Ejemplo de secuencia padded:", x[0])

Shape de entrada (x): (6, 6)
Ejemplo de secuencia padded: [3 4 6 7 0 0]


## Definir modelo
Parametros principales de `Embedding`:
- `input_dim`: tamano de vocabulario (max indice + 1).
- `output_dim`: tamano del vector denso por token.
- `input_length`: longitud fija de cada secuencia de entrada.

In [3]:
vocab_size = len(tokenizer.word_index) + 1
output_dim = 16

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=output_dim, input_length=input_length),
    GlobalAveragePooling1D(),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

print("vocab_size:", vocab_size)
print("output_dim:", output_dim)
print("input_length:", input_length)
model.summary()

vocab_size: 24
output_dim: 16
input_length: 6
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 6, 16)             384       
                                                                 
 global_average_pooling1d (G  (None, 16)               0         
 lobalAveragePooling1D)                                          
                                                                 
 dense (Dense)               (None, 8)                 136       
                                                                 
 dense_1 (Dense)             (None, 1)                 9         
                                                                 
Total params: 529
Trainable params: 529
Non-trainable params: 0
_________________________________________________________________


## Entrenar y observar salida de la capa Embedding
Entrenamos pocas epocas solo para fines didacticos.

In [4]:
history = model.fit(x, etiquetas, epochs=8, verbose=0)
print("Entrenamiento finalizado. Accuracy final aproximada:", round(history.history['accuracy'][-1], 4))

embedding_layer = model.layers[0]
muestra = x[:1]  # Tomamos una secuencia de ejemplo
salida_embedding = embedding_layer(muestra)

print("Shape entrada de muestra:", muestra.shape)
print("Shape salida embedding:", salida_embedding.shape)  # (batch, input_length, output_dim)

Entrenamiento finalizado. Accuracy final aproximada: 0.8333
Shape entrada de muestra: (1, 6)
Shape salida embedding: (1, 6, 16)


## Prediccion sobre una secuencia nueva
Aplicamos el mismo pipeline (tokenizacion + padding) antes de inferir.

In [5]:
nuevo_texto = ["la clase fue muy clara"]
seq_nueva = tokenizer.texts_to_sequences(nuevo_texto)
x_nuevo = pad_sequences(seq_nueva, maxlen=input_length, padding="post", truncating="post")

prob = model.predict(x_nuevo, verbose=0)[0][0]
print("Texto:", nuevo_texto[0])
print("Probabilidad de clase positiva:", round(float(prob), 4))

Texto: la clase fue muy clara
Probabilidad de clase positiva: 0.4972


## Preguntas de reflexion
1. Que efecto tendria duplicar `output_dim`?
2. Que sucede si `input_dim` es menor que el vocabulario real?
3. Por que el output de Embedding tiene 3 dimensiones?